# RADAR API Tutorial (Python) – publish a dataset and correct its metadata

This notebook continues [`radar_api_tutorial.ipynb`](radar_api_tutorial.ipynb). Start there if you are new to the
RADAR API – this notebook only repeats the setup and the token, and explains the steps that are new:

1. Create a dataset with RADAR XML metadata and one file
2. Reserve a DOI and validate the metadata
3. Publish the dataset and wait until it is `PUBLISHED`
4. Correct the metadata of the published dataset and release the correction

Every step shows the Python code and the equivalent `curl` command, so you can transfer it to any language.
The Groovy examples [`publishDataset.groovy`](../groovy/example_metadataCorrection/publishDataset.groovy) and
[`correctDataset.groovy`](../groovy/example_metadataCorrection/correctDataset.groovy) call the same endpoints.

**Prerequisites**
* A RADAR user **with a password** (Shibboleth users are *not* supported)
* An OAuth client (`clientId`, `clientSecret`, `redirectUrl`) – request one at info@radar-service.eu
* The ID of a workspace in which you are allowed to create and publish datasets

> ⚠️ Use the **test** system (`https://test.radar-service.eu`) and a workspace meant for tests.
> **A published dataset cannot be deleted** – neither can its workspace.

## 0. Setup

Install the required package into the Python environment of **this notebook's kernel** (only needed once).

Credentials are **never stored** in this notebook: you are asked for them when you run the cells.

In [ ]:
%pip install --quiet requests

Now import the libraries and define some helpers. Adjust the URLs if you don't use the test system.

In [ ]:
import json, re, time
import xml.etree.ElementTree as ET
from getpass import getpass
from pathlib import Path
from xml.sax.saxutils import escape

import requests

CONTEXT_URL = "https://test.radar-service.eu/radar"
INGEST_URL  = "https://test.radar-service.eu/radar-ingest"

def pretty(data):
    """Print JSON nicely."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

def check(response):
    """Raise an error that includes the server's message if the request failed."""
    if not response.ok:
        raise RuntimeError(f"{response.request.method} {response.url} -> {response.status_code}\n{response.text}")
    return response

### Proxy (optional)

If you need an HTTP proxy to reach RADAR, enter it in the next cell (e.g. `http://proxy.example.org:8080`).
If you leave it empty, your system proxy settings are used. After changing it, restart the kernel.

In [ ]:
import os

PROXY = {
    "HTTP_PROXY":  "",   # e.g. "http://proxy.example.org:8080"
    "HTTPS_PROXY": "",   # usually the same as HTTP_PROXY
    "NO_PROXY":    "",   # e.g. "localhost,radar-service.eu"
}

for name, value in PROXY.items():
    if value:
        os.environ[name] = value    # requests reads the proxy from these environment variables

print("Proxy:", requests.utils.get_environ_proxies(CONTEXT_URL).get("https") or "none (direct connection)")

### Get an OAuth token

The same as in step 1 of the first notebook: `POST /api/tokens` returns the access token for all following requests.

In [ ]:
print("OAuth client credentials")
client_id     = input("  Client ID: ")
client_secret = getpass("  Client secret: ")
redirect_url  = input("  Redirect URL: ")

print("RADAR user credentials")
user_name     = input("  Username: ")
user_password = getpass("  Password: ")

tokens = check(requests.post(f"{CONTEXT_URL}/api/tokens", json={
    "clientId":     client_id,
    "clientSecret": client_secret,
    "redirectUrl":  redirect_url,
    "userName":     user_name,
    "userPassword": user_password,
})).json()
ACCESS_TOKEN = tokens["access_token"]
print("access token:", ACCESS_TOKEN[:8] + "...")

The helper `api()` sends a stateless request with the token. It retries `GET` and `DELETE` on connection problems;
`POST` is never retried, so an action – like publishing – is never executed twice. `wait_for_state()` polls the
dataset until it has reached a state.

In [ ]:
def api(method, url, retries=5, headers=None, **kwargs):
    """Send one stateless API request with the access token. GET and DELETE are retried on connection problems."""
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}", **(headers or {})}
    for attempt in range(retries + 1):
        try:
            return requests.request(method, url, headers=headers, timeout=60, **kwargs)
        except requests.exceptions.ConnectionError:
            if method not in ("GET", "DELETE") or attempt == retries:
                raise
            print(f"connection problem, retrying {method} {url}")
            time.sleep(2 ** attempt)

def dataset_state():
    return check(api("GET", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}")).json()["state"]

def wait_for_state(state, poll_seconds=10, max_polls=90):
    """Poll the dataset until it has reached the given state."""
    for _ in range(max_polls):
        current = dataset_state()
        print("State:", current)
        if current == state:
            return current
        time.sleep(poll_seconds)
    raise TimeoutError(f"Dataset did not reach state {state} within {poll_seconds * max_polls} seconds")

## 1. Create a dataset with metadata and content

These are the steps 4 and 5 of the first notebook in one go:

1. Create the dataset – `POST /api/workspaces/{workspaceId}/datasets`
2. Upload the RADAR XML metadata – `POST /api/datasets/{datasetId}/metadata` (multipart field `metadataFile`).
   Titles must be unique within a workspace, so the title of the example file is replaced.
3. Upload one file – `POST {INGEST_URL}/upload/{datasetId}/file` (multipart field `file`) – and wait until it has
   been processed: `GET {INGEST_URL}/upload/{fileId}/progress` returns `404` once the job has finished.

In [ ]:
WORKSPACE_ID = input("Workspace ID: ")
DATASET_TITLE = f"Dataset published with the Python tutorial {int(time.time() * 1000)}"

dataset = check(api("POST", f"{CONTEXT_URL}/api/workspaces/{WORKSPACE_ID}/datasets", json={
    "id": None,
    "parentId": None,
    "state": "PENDING",
    "technicalMetadata": {
        "retentionPeriod": 10,                                   # years
        "responsibleEmail": "alice@bob.de",
        "publicationBacklink": "http://alice-bob.de/publication",
        "access": "PRIVATE",
        "schema": {"key": "RDDM", "version": "9.3"},
    },
    "descriptiveMetadata": {"title": DATASET_TITLE},
})).json()
DATASET_ID = dataset["id"]
print("Dataset id:", DATASET_ID)

In [ ]:
METADATA_XML = Path("../resources/RadarDatasetExample.xml")
xml = METADATA_XML.read_text(encoding="utf-8")
xml = re.sub(r"(<ns2:title>)[^<]*(</ns2:title>)", lambda m: m[1] + escape(DATASET_TITLE) + m[2], xml, count=1)
pretty(check(api("POST", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/metadata",
                 files={"metadataFile": (METADATA_XML.name, xml.encode("utf-8"), "application/xml")})).json())

sample = Path("sample_data/hello.txt")
with open(sample, "rb") as f:
    FILE_ID = check(api("POST", f"{INGEST_URL}/upload/{DATASET_ID}/file", files={"file": (sample.name, f)})).text.strip()
print("File id:", FILE_ID)

for _ in range(60):
    response = api("GET", f"{INGEST_URL}/upload/{FILE_ID}/progress")
    if response.status_code == 404:         # the server forgets a job once it has finished
        print("Upload processed")
        break
    job = check(response).json()
    print("progress:", job)
    if job["status"] == "completed":
        break
    if job["status"] in {"error", "exception", "stopped", "killed"}:
        raise RuntimeError(f"Upload failed: {job}")
    time.sleep(2)
else:
    raise TimeoutError("Upload was not processed within 2 minutes")

## 2. Reserve a DOI and validate the metadata

A published dataset gets a **DOI**. You can reserve it in advance, while the dataset is still `PENDING` – e.g. to
cite the dataset in a paper before it is published. The answer is the DOI as plain text.

```bash
curl -X POST "${CONTEXT_URL}/api/datasets/${DATASET_ID}/doi" -H "Authorization: Bearer ${ACCESS_TOKEN}"
```

In [ ]:
DOI = check(api("POST", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/doi")).text.strip()
print("Reserved DOI:", DOI)

The DOI is written into the `identifier` of the metadata, which the RADAR schema requires. Validate the metadata
before publishing: any message in `details` is a problem that you have to fix first.

```bash
curl "${CONTEXT_URL}/api/datasets/${DATASET_ID}/metadata/validate" -H "Authorization: Bearer ${ACCESS_TOKEN}"
```

In [ ]:
validation = check(api("GET", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/metadata/validate")).json()
print("success:", validation["success"])
print("details:", validation["details"])

## 3. Publish the dataset – `POST {CONTEXT_URL}/api/datasets/{datasetId}/publish`

```
PENDING ──publish──▶ PUBLISHED
```

⚠️ **A published dataset is public and cannot be changed or deleted anymore** – only its descriptive metadata can
be corrected (step 4). You need the role curator or contract administrator.

```bash
curl -X POST "${CONTEXT_URL}/api/datasets/${DATASET_ID}/publish" -H "Authorization: Bearer ${ACCESS_TOKEN}"
```

Like archiving, publishing runs **asynchronously** in a server job. This takes several minutes, until then the
state stays `PENDING`. The next cell asks you to confirm before it publishes.

In [ ]:
answer = input(f"Publish dataset {DATASET_ID}? It cannot be deleted afterwards. [y/N] ")

if answer.strip().lower() in ("y", "yes"):
    check(api("POST", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/publish"))
    wait_for_state("PUBLISHED")
else:
    print(f"Not published. Run this cell again to publish dataset {DATASET_ID} - the next steps need a published dataset.")

### Read the published metadata – `GET {CONTEXT_URL}/api/datasets/{datasetId}/metadata`

With `Accept: application/xml` you get the descriptive metadata as RADAR XML – the same format you uploaded,
now with the DOI as `identifier`.

```bash
curl "${CONTEXT_URL}/api/datasets/${DATASET_ID}/metadata" \
  -H "Authorization: Bearer ${ACCESS_TOKEN}" -H "Accept: application/xml"
```

In [ ]:
ELEMENTS = "http://radar-service.eu/schemas/descriptive/radar/v09/radar-elements"

def read_metadata():
    """The descriptive metadata of the dataset as RADAR XML text."""
    response = check(api("GET", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/metadata", headers={"Accept": "application/xml"}))
    return response.content.decode("utf-8")

def field(xml, name):
    """The text of a top level element of the RADAR XML, e.g. "title" or "identifier"."""
    return ET.fromstring(xml.encode("utf-8")).findtext(f"{{{ELEMENTS}}}{name}")

metadata = read_metadata()
print("Identifier:", field(metadata, "identifier"))
print("Title     :", field(metadata, "title"))

## 4. Correct the metadata of the published dataset

A typo in the title, a missing co-author, a new funding reference: the descriptive metadata of a published
dataset can be corrected. The files cannot. A correction takes two steps:

1. **Upload** the corrected metadata – `POST {CONTEXT_URL}/api/datasets/{datasetId}/metadata-correction`
   (multipart field `metadataFile`). It is stored as a draft, the published metadata does not change yet.
2. **Release** the correction – `POST {CONTEXT_URL}/api/datasets/{datasetId}/metadata-correction/release`.
   The corrected metadata replaces the published one.

The correction **replaces** the whole descriptive metadata. So start from the published XML of step 3 and change
only what needs correcting. Keep the `identifier`: it is the DOI of the dataset.

Here we correct the title. We replace it in the XML text, so everything else stays exactly as it was published:

In [ ]:
CORRECTED_TITLE = DATASET_TITLE + " (corrected)"

corrected_xml, count = re.subn(r"(<(?:\w+:)?title(?:\s[^>]*)?>)[^<]*(</(?:\w+:)?title>)",
                               lambda m: m[1] + escape(CORRECTED_TITLE) + m[2], read_metadata(), count=1)
assert count == 1, "no <title> found in the metadata"
print("Title:", field(corrected_xml, "title"))

### 4a. Upload the correction

```bash
curl -X POST "${CONTEXT_URL}/api/datasets/${DATASET_ID}/metadata-correction" \
  -H "Authorization: Bearer ${ACCESS_TOKEN}" -F "metadataFile=@corrected.xml"
```

This only works for a `PUBLISHED` dataset.

In [ ]:
response = check(api("POST", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/metadata-correction",
                     files={"metadataFile": ("corrected.xml", corrected_xml.encode("utf-8"), "application/xml")}))
print(response.status_code, response.text)

### 4b. Release the correction

```bash
curl -X POST "${CONTEXT_URL}/api/datasets/${DATASET_ID}/metadata-correction/release" \
  -H "Authorization: Bearer ${ACCESS_TOKEN}"
```

In [ ]:
response = check(api("POST", f"{CONTEXT_URL}/api/datasets/{DATASET_ID}/metadata-correction/release"))
print(response.status_code, response.text)

### 4c. Check the result

Read the metadata again until the corrected title shows up. The DOI is the same as before.

In [ ]:
for _ in range(30):
    metadata = read_metadata()
    if field(metadata, "title") == CORRECTED_TITLE:
        break
    time.sleep(10)
else:
    raise TimeoutError("The corrected title did not show up within 5 minutes")

print("Identifier:", field(metadata, "identifier"))
print("Title     :", field(metadata, "title"))
print("State     :", dataset_state())

## Where to go next

* Open your dataset in the RADAR web UI: the landing page shows the corrected metadata.
* Datasets published on the production system can be resolved at `https://doi.org/<DOI>`.
* The RADAR XML schema: <https://www.radar-service.eu/schemas/descriptive/radar/v09/radar-dataset>

---
Copyright 2026 FIZ Karlsruhe - Leibniz-Institut fuer Informationsinfrastruktur GmbH

Licensed under the [Apache License, Version 2.0](../LICENSE).
